# J2 Après-Midi | Les ingénieurs du ... prompt

**Objectifs d'apprentissage :**
1. Prise en main l'API d'un Modèle de Langage.
2. Comprendre les principes du *Prompt Engineering* (Zero-shot, Few-shot, Chain of Thought).
3. Comparer les performances (VADER vs Modèle Local HuggingFace vs API LLM).
4. Mettre en place une validation humaine (inter-coder agreement) pour vérifier les résultats.
5. Appliquer ces concepts pour annoter des jeu de données.


## 1. Introduction à l'API OpenAI

Lorsqu'on utilise ChatGPT via le site web, l'interface graphique envoie en arrière-plan nos requêtes à l'API d'OpenAI. Nous allons automatiser cela directement en Python.

**Attention !** En temps normal, ne mettez **jamais** votre clé API directement dans le code public. Pour les besoins de cet atelier, vous allez copier-coller temporairement la clé fournie par les instructeurs directement dans la variable `api_key`.

In [ ]:
from openai import OpenAI
import pandas as pd

# 1. Coller votre clé API d'OPEN AI
api_key = "sk-votre-cle-secrete-ici"

try:
    client = OpenAI(api_key=api_key)
    print("Client OpenAI initialisé avec succès !")
except:
    print("Erreur d'initialisation. Vérifiez votre clé API.")


In [ ]:
# Warm-up
text = "Peux-tu m'expliquer le prompt engineering très simplement ?"

client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        # {"role": "system", "content": "Vous êtes un assistant utile."},
        {"role": "user", "content": text}
    ],
    # temperature 0 = prévisible/déterministe, 1 = créatif/aléatoire
    temperature=1,  
    max_tokens=100,
)

### Hack-Time 🛠️

Observez la structure compliquée (JSON) de la réponse ci-dessus. 
**Exercice :** Assignez la réponse à une variable `response`, puis essayez d'isoler et d'afficher uniquement le texte généré par l'IA.

In [ ]:
# Votre code ici


### Création d'une fonction réutilisable (DRY)

Pour éviter d'avoir à réécrire tout cela, nous avons préparé une fonction `analyze_text` que nous utiliserons pour la suite de l'atelier.

In [ ]:
def analyze_text(text, system_prompt, model="gpt-4o-mini", temperature=0.0):
    """
    Fonction pour envoyer un texte et un prompt système à l'API OpenAI.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": text}
            ],
            # temperature 0 = prévisible/déterministe, 1 = créatif/aléatoire
            temperature=temperature,  
            max_tokens=150
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Erreur lors de l'appel à l'API : {e}"

## 2. L'impact du Prompt Engineering sur la Performance

En sciences sociales, **le prompt est l'équivalent d'un protocole d'annotation (codebook)**. Si le prompt est ambigu, le codage de vos données sera de mauvaise qualité.

Pour observer concrètement l'impact de l'amélioration des prompts, nous allons utiliser un jeu de données de test de 25 phrases politiques, complexes à analyser (ironie, négation, nuance, style journalistique).
Nous évaluerons si l'IA trouve le "vrai" label (Positif, Négatif ou Neutre).

In [ ]:
# Création de notre jeu de données (25 phrases)
test_data = [
    {"text": "Cette réforme est une excellente opportunité pour notre économie.", "label": "POSITIF"},
    {"text": "Nous soutenons pleinement cette initiative très prometteuse.", "label": "POSITIF"},
    {"text": "Le projet de loi est un désastre total et une honte.", "label": "NEGATIF"},
    {"text": "Je suis extrêmement déçu par ces mesures injustes.", "label": "NEGATIF"},
    {"text": "Le ministre a annoncé la date des prochaines élections.", "label": "NEUTRE"},
    {"text": "Quelle brillante idée d'augmenter les taxes de 20%, un pur coup de génie !", "label": "NEGATIF"},
    {"text": "Bravo au gouvernement pour cette inflation insupportable, on adore !", "label": "NEGATIF"},
    {"text": "Un triomphe absolu que d'avoir réussi à faire grève pendant 3 mois...", "label": "NEGATIF"},
    {"text": "Une manifestation violente a rassemblé 10 000 personnes en colère contre le gouvernement.", "label": "NEUTRE"},
    {"text": "Le scandale de corruption a éclaté ce matin à la Une des journaux.", "label": "NEUTRE"},
    {"text": "La crise économique provoque la fermeture de nombreuses entreprises dans la région.", "label": "NEUTRE"},
    {"text": "L'accord historique a été signé dans un climat de tension extrême.", "label": "NEUTRE"},
    {"text": "Le député a attaqué violemment le bilan de son adversaire lors du débat.", "label": "NEUTRE"},
    {"text": "Il serait très exagéré de dire que cette mesure est mauvaise.", "label": "POSITIF"},
    {"text": "Je ne peux pas m'empêcher de penser que ce n'est pas complètement un échec.", "label": "POSITIF"},
    {"text": "Pas exactement le succès du siècle.", "label": "NEGATIF"},
    {"text": "Rien ne prouve que ce texte soit une erreur historique.", "label": "POSITIF"},
    {"text": "On a déjà vu pire comme proposition de loi.", "label": "POSITIF"},
    {"text": "La loi autorise la fermeture des commerces le dimanche.", "label": "NEUTRE"},
    {"text": "Le président a annulé sa visite prévue dans les territoires sinistrés.", "label": "NEUTRE"},
    {"text": "Les impôts augmenteront de 2% pour financer la sécurité sociale.", "label": "NEUTRE"},
    {"text": "Le texte final est inapplicable bien qu'il y ait quelques bonnes idées.", "label": "NEGATIF"},
    {"text": "Un débat long et ennuyeux, mais qui accouche d'une mesure très positive.", "label": "POSITIF"},
    {"text": "Malgré les protestations, l'assemblée a validé un texte fondamental pour nos libertés.", "label": "POSITIF"},
    {"text": "C'est un véritable triomphe historique pour l'opposition qui a su imposer ses idées.", "label": "POSITIF"}
]

test_df = pd.DataFrame(test_data)
test_df.head()


### A. Les approches classiques et locales (Rappels)

Testons d'abord **VADER** (basé sur un dictionnaire de mots) et le modèle local **DistilCamembert** (vu précédemment).

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings('ignore')

nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

def evaluate_vader(text):
    score = sia.polarity_scores(text)['compound']
    if score > 0.1: return "POSITIF"
    elif score < -0.1: return "NEGATIF"
    else: return "NEUTRE"

test_df["vader"] = test_df["text"].apply(evaluate_vader)


In [ ]:
from transformers import pipeline

local_analyzer = pipeline("sentiment-analysis", model="cmarkea/distilcamembert-base-sentiment")

def evaluate_hf(text):
    res = local_analyzer(text)[0]['label']
    if '1' in res or '2' in res: return "NEGATIF"
    elif '4' in res or '5' in res: return "POSITIF"
    else: return "NEUTRE"

test_df["camembert"] = test_df["text"].apply(evaluate_hf)


> **Observation** : VADER et Camembert échouent souvent sur l'ironie ou sur des formulations complexes. Leurs scores de bonnes réponses (accuracy) plafonnent à ~40% sur ce jeu de données complexe.

### B. Zero-Shot Prompting
C'est la méthode la plus simple. On donne l'instruction sans fournir d'exemple préalable. Vous constaterez que même un LLM très puissant, s'il est mal guidé, fera des erreurs (surtout sur le journalisme qu'il prendra pour du négatif).

In [ ]:
zero_shot_prompt = """
Analyse le sentiment du texte suivant.
Catégories possibles : POSITIF, NEGATIF, NEUTRE.
Réponds UNIQUEMENT par la catégorie (sans ponctuation).
"""

test_df["zero_shot"] = test_df["text"].apply(lambda x: analyze_text(x, zero_shot_prompt))


### C. Few-Shot Prompting
Fournir des exemples (*Few-Shot*) agit comme un **livre de codage (codebook)** pour l'IA.

In [ ]:
few_shot_prompt = """
Analyse le sentiment du texte.
Catégories possibles : POSITIF, NEGATIF, NEUTRE.

Attention :
- L'ironie compte souvent comme NEGATIF.
- Les rapports de faits (journalisme) comptent comme NEUTRE, même s'ils décrivent de la violence ou une crise.

Exemples :
Texte : "Génial, encore une taxe de plus..."
Catégorie : NEGATIF
Texte : "La police a arrêté 50 manifestants."
Catégorie : NEUTRE

À toi : 
Réponds UNIQUEMENT par la catégorie.
"""

test_df["few_shot"] = test_df["text"].apply(lambda x: analyze_text(x, few_shot_prompt))


### D. Chain of Thought (Chaîne de Pensée)
Demander au modèle de justifier son choix *avant* de donner la réponse finale l'aide à mieux gérer l'ironie et les retournements de situation.

In [ ]:
cot_prompt = """
Tu dois analyser l'orientation (sentiment) exprimée par l'auteur du texte.
Catégories : POSITIF, NEGATIF, NEUTRE.

Règles de codage :
- POSITIF : L'auteur soutient ou loue l'idée.
- NEGATIF : L'auteur critique, ironise ou dénigre.
- NEUTRE : L'auteur se contente de rapporter des faits (journalisme, description), même si le sujet est grave (crise, violence, impôts).

Processus :
1. Raisonnement : Cherche l'intention de l'auteur. Est-ce descriptif ou expressif ? Y a-t-il des doubles négations ou de l'ironie ?
2. Réponse finale : Tu dois terminer par exactement "LABEL: POSITIF", "LABEL: NEGATIF" ou "LABEL: NEUTRE".
"""

def extract_label(cot_response):
    if "LABEL: POSITIF" in cot_response.upper(): return "POSITIF"
    if "LABEL: NEGATIF" in cot_response.upper(): return "NEGATIF"
    if "LABEL: NEUTRE" in cot_response.upper(): return "NEUTRE"
    return "ERREUR"

test_df["cot_reasoning"] = test_df["text"].apply(lambda x: analyze_text(x, cot_prompt))
test_df["cot"] = test_df["cot_reasoning"].apply(extract_label)


### E. Comparaison des performances

Calculons le pourcentage de bonnes réponses (accuracy) pour chaque approche et visualisons l'évolution.

In [ ]:
# Calcul des performances
performance = (
    test_df[["vader", "camembert", "zero_shot", "few_shot", "cot"]]
    .eq(test_df["label"], axis=0)
    .mean()
)

print("--- Accuracy par méthode sur 25 phrases ---")
performance 

In [ ]:
# On peut faire de magnifiques graphique d'évaluation 
performance.plot.bar()

In [ ]:
# Mais ...

In [ ]:
# Affichage des erreurs de Zero-Shot pour comprendre pourquoi il échoue
zs_errors = test_df[test_df["zero_shot"] != test_df["label"]][["text", "label", "zero_shot"]]
zs_errors


## 3. Validation Humaine (Inter-coder Reliability)

Attention à ne jamais aveuglément faire confiance à un modèle... Il est indispensable de coder humainement un sous-échantillon pour vérifier que vous êtes d'accord avec l'IA. 
Pour aller plus loin qu'un simple pourcentage, nous allons calculer le **Kappa de Cohen**, une métrique statistique standard en sciences sociales pour mesurer l'accord inter-juges.

In [ ]:
# 1. Vos données à coder
sentences_to_code = [
    "La réforme des retraites est un désastre social sans précédent.",
    "Le texte a été voté par 349 voix pour et 86 contre.",
    "Une avancée majeure pour les droits des travailleurs a été obtenue hier soir.",
    "La manifestation de ce mardi a bloqué le centre-ville.",
    "Incroyable talent de notre ministre pour faire couler l'économie..."
]


In [ ]:
# 2. Entrez vos labels humains ici ! 
# Remplacez "..." par "POSITIF", "NEGATIF" ou "NEUTRE"
human_labels = [
    "NEGATIF", # Phrase 1
    "NEUTRE",  # Phrase 2
    "POSITIF", # Phrase 3
    "NEUTRE",  # Phrase 4
    "NEGATIF"  # Phrase 5 
]


In [ ]:
# 3. L'IA code les mêmes phrases (avec le meilleur prompt, CoT)
ai_labels = []
for phrase in sentences_to_code:
    rep = analyze_text(phrase, cot_prompt)
    ai_labels.append(extract_label(rep))


In [ ]:
# 4. Comparaison
validation_df = pd.DataFrame({
    "text": sentences_to_code,
    "human_label": human_labels,
    "ai_label": ai_labels
})

validation_df["agreement"] = validation_df["human_label"] == validation_df["ai_label"]
validation_df


In [ ]:
from sklearn.metrics import cohen_kappa_score

agreement_rate = validation_df['agreement'].mean() * 100
kappa_score = cohen_kappa_score(validation_df["human_label"], validation_df["ai_label"])

print(f"Taux d'accord Humain vs IA : {agreement_rate}%")
print(f"Kappa de Cohen : {kappa_score:.2f} (1.0 = Accord parfait, > 0.8 = Excellent)")


> **Métho?** : Dans un vrai papier de recherche, un Kappa supérieur à 0.80 est généralement considéré comme très solide. En dessous de 0.60, il faut retravailler vos instructions (prompts) car l'accord est trop faible.

## 4. Atelier de Codage : Le Projet INA / ZFE

Appliquons maintenant ce que nous avons appris sur notre cas d'usage réel : les "Zones à Faibles Émissions" (ZFE) sur CNews et TF1.

In [ ]:
ina_df = pd.read_csv("https://raw.githubusercontent.com/mickaeltemporao/lillelms/refs/heads/main/data/ina_zfe/ina_zfe.csv")
ina_df.head()


Notre objectif de recherche : **Le titre de l'émission est-il favorable, défavorable, ou neutre vis-à-vis de la politique environnementale des ZFE ?**

In [ ]:
zfe_system_prompt = """
Ton expertise est en sociologie des médias.
Ton but est d'annoter l'orientation d'un titre de journal télévisé vis-à-vis des politiques environnementales (ZFE).

Classe le titre selon 3 catégories :
- FAVORABLE : Le titre présente les ZFE comme une bonne chose, une solution, un progrès.
- DEFAVORABLE : Le titre souligne la colère, les problèmes, les interdictions, les coûts ou le côté punitif.
- NEUTRE : Le titre est purement descriptif et factuel, sans jugement de valeur.

Réponds UNIQUEMENT par la catégorie (FAVORABLE, DEFAVORABLE, NEUTRE).
"""


### Hack-Time 

1. **Testez Camembert** : Utilisez la fonction `evaluate_hf` (vue plus haut) sur ces titres de l'INA. Observez-vous des erreurs de la part du modèle local ?
2. **Ajustez les Prompts** : Complétez la boucle ci-dessous pour envoyer chaque titre de l'émission à l'API OpenAI avec notre `zfe_system_prompt`. 
3. **Comparez** : L'IA d'OpenAI fait-elle mieux que Camembert ? Si non, transformez `zfe_system_prompt` en prompt **Few-Shot** ou **Chain of Thought** pour forcer l'IA à analyser la ligne éditoriale du média.

In [ ]:
ina_results = []

for index, row in ina_df.head(5).iterrows():
    title = row['titre']
    channel = row['chaine']
    
    # 1. Test avec Camembert local
    camembert_score = evaluate_hf(title)
    
    # 2. Appel à l'API avec le prompt
    llm_annotation = analyze_text(title, zfe_system_prompt)
    
    print(f"[{channel}] {title} \n  Camembert -> {camembert_score} \n  LLM OpenAI -> {llm_annotation}\n")
    ina_results.append(llm_annotation)


## 5. Analyse sur Bluesky

Pour terminer, lions l'extraction par API (vue ce matin) et l'analyse par LLM.
Nous allons extraire dynamiquement les ~25 derniers messages de personnalités politiques sur Bluesky, puis utiliser l'API OpenAI pour en analyser le ton.

In [ ]:
import requests

# 1. Collecte via l'API publique de Bluesky (sans clé API)
politicians = [
    "jlmelenchon.bsky.social",
    "olivierfaure.bsky.social"
]

bluesky_data = []

for pol in politicians:
    # On limite à 10 messages par compte (soit ~20 messages au total)
    url = f"https://public.api.bsky.app/xrpc/app.bsky.feed.getAuthorFeed?actor={pol}&limit=10"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        for item in data.get('feed', []):
            post = item.get('post', {}).get('record', {})
            text = post.get('text', '')
            # On ignore les messages vides (ex: juste une image)
            if text:
                bluesky_data.append({
                    "politician": pol.split('.')[0],
                    "text": text
                })

bluesky_df = pd.DataFrame(bluesky_data)
print(f"{len(bluesky_df)} messages récupérés.")
bluesky_df.head()


### Hack-Time 🛠️

Votre mission :
1. Créez un `bluesky_prompt` pour catégoriser les messages (par exemple : `POLEMIQUE`, `CONSTRUCTIF`, ou `NEUTRE`).
2. Itérez pour envoyer chaque texte à la fonction `analyze_text`.
3. Explorez les résultats. Le modèle arrive-t-il à capter la nuance des discours politiques réels ?

In [ ]:
# 1. Rédigez votre prompt
bluesky_prompt = """
Tu es un politiste analysant le ton de messages sur les réseaux sociaux.
... (À compléter) ...
"""


In [ ]:
# 2. Analyse
bluesky_df["annotation"] = bluesky_df["text"].apply("")


In [ ]:
# 3. Exploration...

## 6. Conclusion

Nous avons vu comment utiliser le Prompt Engineering pour améliorer la qualité de l'annotation automatique, et comment valider ces résultats.

**Principes de recherche à retenir :**
1. **Tester différentes approches** : Zero-Shot, Few-Shot, CoT. Choisissez celles qui maximisent les performances sur un échantillon de validation.
2. **Biais du Modèle :** Les modèles comportent les mêmes biais qu'on voit autour de nous. Attention à ne pas reproduire cela à grande échelle.
3. **Reproductibilité :** Si l'API met à jour son modèle, vos résultats dans 6 mois pourraient différer. Conservez toujours vos prompts exacts et les versions des modèles (ex: `gpt-4o-mini-2024-07-18`) dans vos scripts !
4. **Validation Intercodeur:** Ne faites pas confiance à la machine. L'étape de codage humain sur un échantillon (et le calcul du Kappa) est un minimum scientifique.